# DAI Mission — Proposal Template
**Data & AI in Economics | TU Dortmund**

This notebook is your team's mission proposal. Fill in every section before submission. Once approved, you will extend this same notebook into your final deliverable.

> **Team size:** 2–3 students  
> **Deliverable:** This Jupyter Notebook (proposal → final submission in one file)


## 1. Team

| Role | Name |
|------|------|
| Lead | Oliver Ossadnik |
| Member | Danish Ahmad |


## 2. Mission Title & Research Question

**Title:** Clearing the Air: Evaluating Causal Policy Impacts, Meteorological Regimes, and Machine Learning Predictors of PM2.5 Pollution Across Five Chinese Macroeconomic Hubs

**Research question:** To what extent does the mandatory winter-heating policy in Northern Chinese cities (Beijing and Shenyang) causally increase hourly PM2.5 concentrations compared to Southern cities (Shanghai, Guangzhou, Chengdu) when adjusting for meteorological confounding factors, and how accurately can non-linear supervised learning models forecast these fluctuations using structural environmental features?

**Why it matters:** Air pollution acts as a severe negative economic externality of rapid industrialization, inflicting substantial public health costs, elevating mortality rates, and diminishing urban worker productivity. Disentangling the structural impact of regional public utility policies (such as centralized winter heating infrastructure) from purely natural meteorological variations is vital for designing sustainable urban regulations and optimizing green energy transitions without derailing regional economic growth.


## 3. Data

**Source(s):** * PM2.5 Data of Five Chinese Cities Dataset, hosted by the UCI Machine Learning Repository.
* URL: https://archive-beta.ics.uci.edu/dataset/394/pm2+5+data+of+five+chinese+cities
* Data Pedigree: Liang, X., Li, S., Zhang, S., Huang, H., and Chen, S. X. (2016). PM2.5 data reliability, consistency, and air quality assessment in five Chinese cities. Journal of Geophysical Research: Atmospheres, 121(17), 10,220-10,236.

**Unit of observation:** An hourly meteorological and air quality observation for a given city between January 1st, 2010, and December 31st, 2015.

**Key variables:**

| Variable | Type | Role (feature / target / instrument / ...) | Description |
|----------|------|---------------------------------------------|-------------|
| `PM` | Continuous | Target (Supervised) / Outcome (Causal) | PM2.5 concentration ($\mu g/m^3$) averaged across urban monitoring stations. |
| `heating_regime` | Binary (0 or 1) | Treatment Variable (Causal) | Engineered indicator: 1 if city is in the winter-heating region (Beijing/Shenyang) during active heating months (Nov-March), 0 otherwise. |
| `TEMP` | Continuous | Feature / Confounder | Ambient temperature in Celsius. |
| `DEWP` | Continuous | Feature / Confounder | Dew Point temperature in Celsius. |
| `HUMI` | Continuous | Feature / Confounder | Humidity percentage (%). |
| `PRES` | Continuous | Feature / Confounder | Atmospheric pressure (hPa). |
| `Iws` | Continuous | Feature / Confounder | Cumulated wind speed (m/s). |
| `city` | Categorical | Feature / Context | The specific metropolitan center (Beijing, Shanghai, Guangzhou, Chengdu, Shenyang). |

**Potential data quality issues:** * **Missing Values (`NA`):** The target variable `PM` contains missing measurements during specific hours due to station equipment maintenance, requiring systematic forward-fill imputation.
* **Latent Missing Meteorological Fields:** Initial processing revealed hidden `NaN` entries in key weather parameters (`DEWP`, `Iws`) across specific years. Because machine learning models like K-Means clustering natively reject missing attributes, a strict drop-row filtration pipeline must be established across all core modeling features.


In [1]:
# Data loading & first inspection
# ────────────────────────────────────────────────────────────────────────
import os
import pandas as pd
import numpy as np

# 1. Define paths to your 5 uploaded city datasets
file_mapping = {
    'Beijing': 'BeijingPM20100101_20151231.csv',
    'Shanghai': 'ShanghaiPM20100101_20151231.csv',
    'Guangzhou': 'GuangzhouPM20100101_20151231.csv',
    'Chengdu': 'ChengduPM20100101_20151231.csv',
    'Shenyang': 'ShenyangPM20100101_20151231.csv'
}

combined_dfs = []

print("Ingesting and combining city datasets...")
for city, filename in file_mapping.items():
    if not os.path.exists(filename):
        print(f"Warning: File {filename} not found. Please ensure it is in this folder.")
        continue
        
    # Read file and parse common string representations of missing data
    df = pd.read_csv(filename, na_values=['NA', 'NaN', ' '])
    df['city'] = city
    
    # Consolidate city-specific monitoring station columns into a single average target column
    pm_cols = [col for col in df.columns if col.startswith('PM_')]
    df['PM'] = df[pm_cols].mean(axis=1)
    
    # Extract only the structural columns shared across the target models
    core_cols = ['city', 'year', 'month', 'day', 'hour', 'season', 'PM', 
                 'TEMP', 'PRES', 'HUMI', 'DEWP', 'Iws', 'precipitation']
    existing_cols = [c for c in core_cols if c in df.columns]
    
    combined_dfs.append(df[existing_cols])

# 2. Compile into a master panel DataFrame
df = pd.concat(combined_dfs, ignore_index=True)

# 3. Handle missing data to prevent downstream ML crashes (Forward fill timeline gaps, drop rows with NaN weather details)
df = df.sort_values(by=['city', 'year', 'month', 'day', 'hour']).reset_index(drop=True)
df['PM'] = df.groupby('city')['PM'].ffill()
df = df.dropna(subset=['PM', 'TEMP', 'PRES', 'HUMI', 'DEWP', 'Iws']).reset_index(drop=True)

print(f"Master dataset successfully initialized. Total observations: {df.shape[0]} rows\n")

# 4. First inspections requested by template
print("--- FIRST 5 ROWS ---")
display(df.head())

print("\n--- DATASET SUMMARY / STRUCTURE ---")
df.info()

print("\n--- STATISTICAL DISTRIBUTION OF VARIABLES ---")
display(df.describe())

Ingesting and combining city datasets...
Master dataset successfully initialized. Total observations: 180979 rows

--- FIRST 5 ROWS ---


,city,year,month,day,hour,season,PM,TEMP,PRES,HUMI,DEWP,Iws,precipitation
0,Beijing,2010,1,1,23,4.0,129.0,-5.0,1020.0,41.0,-17.0,0.89,0.0
1,Beijing,2010,1,2,0,4.0,148.0,-4.0,1020.0,38.0,-16.0,1.79,0.0
2,Beijing,2010,1,2,1,4.0,159.0,-4.0,1020.0,42.0,-15.0,2.68,0.0
3,Beijing,2010,1,2,2,4.0,181.0,-5.0,1021.0,63.5,-11.0,3.57,0.0
4,Beijing,2010,1,2,3,4.0,138.0,-5.0,1022.0,85.0,-7.0,5.36,0.0



--- DATASET SUMMARY / STRUCTURE ---
<class 'pandas.DataFrame'>
RangeIndex: 180979 entries, 0 to 180978
Data columns (total 13 columns):
 #   Column         Non-Null Count   Dtype  
---  ------         --------------   -----  
 0   city           180979 non-null  str    
 1   year           180979 non-null  int64  
 2   month          180979 non-null  int64  
 3   day            180979 non-null  int64  
 4   hour           180979 non-null  int64  
 5   season         180979 non-null  float64
 6   PM             180979 non-null  float64
 7   TEMP           180979 non-null  float64
 8   PRES           180979 non-null  float64
 9   HUMI           180979 non-null  float64
 10  DEWP           180979 non-null  float64
 11  Iws            180979 non-null  float64
 12  precipitation  175487 non-null  float64
dtypes: float64(8), int64(4), str(1)
memory usage: 19.3 MB

--- STATISTICAL DISTRIBUTION OF VARIABLES ---


,year,month,day,hour,season,PM,TEMP,PRES,HUMI,DEWP,Iws,precipitation
count,180979.000000,180979.000000,180979.000000,180979.000000,180979.000000,180979.000000,180979.000000,180979.000000,180979.000000,180979.000000,180979.000000,175487.000000
mean,2013.292354,6.617779,15.776460,11.500898,2.504362,72.757898,15.911110,1013.595769,66.150129,8.429249,21.792738,0.123380
std,1.394203,3.451543,8.815972,6.921839,1.110155,68.365250,11.094051,10.029801,52.545431,48.770160,46.482144,1.169719
min,2010.000000,1.000000,1.000000,0.000000,1.000000,1.000000,-27.000000,975.000000,-9999.000000,-9999.000000,0.000000,0.000000
25%,2012.000000,4.000000,8.000000,6.000000,2.000000,28.000000,8.300000,1006.000000,50.920000,0.300000,2.000000,0.000000
50%,2013.000000,7.000000,16.000000,12.000000,3.000000,52.000000,18.000000,1013.000000,70.360000,11.800000,5.200000,0.000000
75%,2014.000000,10.000000,23.000000,18.000000,3.000000,92.666667,24.700000,1021.000000,86.000000,19.000000,19.000000,0.000000
max,2015.000000,12.000000,31.000000,23.000000,4.000000,1134.666667,42.000000,1046.000000,100.000000,28.000000,691.000000,90.400002


## 4. Planned Methods

### 4a. Causal Inference
- [x] Causal graph / DAG (DoWhy)
- [x] Backdoor adjustment

*Justification:*
Weather patterns directly determine localized heating demand (treatment) and alter air pollution dispersion capabilities (outcome). By building an explicit Directed Acyclic Graph (DAG) via the `DoWhy` framework, we can map structural assumptions and execute Backdoor Adjustment to block meteorological confounders (`TEMP`, `PRES`, `HUMI`), allowing us to isolate the true causal effect of the urban heating policy on PM2.5 levels.

### 4b. Supervised Learning
- [x] Decision Tree / Random Forest

*Justification:*
Atmospheric chemical configurations involve highly complex, non-linear physical thresholds (e.g., wind speed and temperature must pass specific boundaries to disperse smog). Random Forests are structurally optimized to capture these threshold phenomena and multi-variable interactions without forcing strict linear assumptions.

### 4c. Unsupervised Learning / Generative Models
- [x] K-Means clustering

*Justification:*
We will apply K-Means clustering to the clean meteorological features (`TEMP`, `DEWP`, `HUMI`, `PRES`, `Iws`) to discover latent 'Atmospheric Weather Regimes' across the cities. This helps economists determine if policy externalities are amplified under specific natural micro-climate profiles.


## 5. Evaluation Strategy

To ensure empirical validity, statistical robustness, and to protect against overfitting or spurious correlations, our mission will deploy a multi-tiered evaluation framework tailored specifically to each of the three analytical methodologies.

### 5a. Causal Inference Validation & Policy Robustness
Because causal claims cannot be verified using simple out-of-sample prediction metrics, we will evaluate our Average Treatment Effect (ATE) estimate through structural econometric diagnostics and algorithmic refutation frameworks using the `DoWhy` architecture:

1. **Statistical Inference and Panel Adjustments:** Rather than relying on naive standard errors, we will compute robust, clustered standard errors. Because air quality data exhibits strong spatial and temporal autocorrelation (shocks to air quality in one hour persist into subsequent hours), we will evaluate the statistical significance ($p$-values and 95% confidence intervals) by bootstrapping standard errors clustered at the `city` and `month` levels.
2. **Placebo Treatment Refutation:** We will run a structural validation test where the true policy variable (`heating_regime`) is replaced with an artificially generated, random assignment vector. If our backdoor adjustment graph is correctly specified and has blocked all active confounders, the newly calculated placebo treatment effect must collapse to 0 ($p > 0.05$). Any statistically significant non-zero effect will indicate the presence of unobserved target leakage or structural DAG misspecification.
3. **Random Common Cause Addition:** We will programmatically introduce an artificial independent random variable into our dataset as a simulated confounder. We will then rerun the backdoor linear regression estimation pipeline. The estimated causal coefficient for the winter heating policy must remain stable and invariant to this addition, confirming that our model is resilient to changes in atmospheric noise features.
4. **Data Subsets Refuter:** We will randomly drop 10% to 20% slices of our master panel data matrix and re-estimate the policy coefficient. The distribution of the resulting coefficients must remain tightly grouped around our primary ATE estimate, demonstrating that our causal insights are not driven by extreme seasonal outliers or anomaly hours in any single city.

### 5b. Supervised Learning Generalization Capacity
To assess how effectively our non-linear Random Forest Regressor models the transformation of physical weather matrices into air pollution, we will evaluate its performance using a rigorous validation pipeline:

1. **Data Splitting Strategy:** To simulate true out-of-sample forecasting, we will implement a chronological, time-based split rather than a completely randomized shuffle split. We will train our model on historical data spanning 2010 through 2014, and preserve the entire 2015 calendar year across all five cities as a strict holdout testing set. This ensures our evaluation strategy accounts for non-stationary trends, macro climate variations, and structural economic adjustments over time.
2. **Statistical Metrics:** Performance on the holdout test set will be quantified using three distinct metrics:
   * **Root Mean Squared Error (RMSE):** Measures prediction error magnitude while heavily penalizing large deviations, which is crucial because extreme, unexpected smog spikes carry the highest public health and economic costs ($\text{RMSE} = \sqrt{\frac{1}{n}\sum(y_i - \hat{y}_i)^2}$).
   * **Mean Absolute Error (MAE):** Provides a robust measure of median deviation, indicating the expected typical error margin during standard hourly operations ($\text{MAE} = \frac{1}{n}\sum|y_i - \hat{y}_i|$).
   * **Coefficient of Determination ($R^2$):** Measures the proportion of total hourly PM2.5 variance that can be successfully explained by our meteorological and policy features ($R^2 = 1 - \frac{SS_{res}}{SS_{tot}}$).
3. **Baseline Comparison:** We will establish a clear benchmark for success by comparing our Random Forest model against a baseline Ordinary Least Squares (OLS) linear model and a naive lag-1 persistence model (predicting that the current hour's pollution equals the previous hour's pollution). Our machine learning model will be considered successful if it achieves a minimum 15% reduction in out-of-sample RMSE compared to these baseline benchmarks.

### 5c. Unsupervised Learning Profiling and Clustering Metrics
Because clustering lacks explicit target labels, we will utilize a blend of internal geometric metrics and external domain-driven evaluations to confirm the validity of our weather regimes:

1. **Geometric Optimization (The Elbow Method):** We will evaluate the internal compactness and separation of our climate groups by computing the **Within-Cluster Sum of Squares (Inertia)** across a range of potential cluster sizes ($K \in [2, 8]$). The optimal number of regimes ($K=4$) will be verified by identifying the structural "elbow" point where adding further clusters yields diminishing returns in variance explanation.
2. **Feature Scaler Validation:** Since clustering relies on Euclidean distance measurements, variables with larger absolute scales (such as atmospheric pressure, `PRES` $\approx 1000+$ hPa) could mathematically overwhelm smaller variables (such as temperature, `TEMP`). We will verify that our `StandardScaler` preprocessing pipeline achieves a uniform mean of 0 and variance of 1 across all features before clustering occurs.
3. **External Domain Validation (Target Separation Check):** While our K-Means model is trained strictly on natural weather features (`TEMP`, `DEWP`, `HUMI`, `PRES`, `Iws`) without ever seeing air pollution values, we will evaluate its real-world economic utility by mapping the *historical PM2.5 distributions* back onto each assigned cluster ID. If the unsupervised clusters reveal statistically distinct distributions of air quality (e.g., one cluster systematically groups severe smog events while another captures clean air days), it proves the algorithm has successfully extracted meaningful, latent environmental macro-regimes.


## 6. Work Plan

| Step | Owner | Description |
|------|-------|-------------|
| **1. Data collection & cleaning** | Danish | Code data ingestion across all 5 CSV files. Resolve latent missing entries (`NaN`) in target (`PM`) and meteorological features (`DEWP`, `Iws`) using forward-fill and strict drop-row filtration to ensure clean inputs for downstream algorithms. |
| **2. EDA** | Danish | Perform detailed Exploratory Data Analysis. Standardize climate features using `StandardScaler` to prevent scale dominance. Map regional urban boundaries to engineer the structural binary `heating_regime` policy indicator variable and plot baseline cross-city pollution trends. |
| **3. Causal inference block** | Oliver | Construct the explicit Directed Acyclic Graph (DAG) using the `DoWhy` framework. Define backdoor criteria to isolate the True Average Treatment Effect (ATE) of winter heating, adjusting for weather confounders (`TEMP`, `PRES`, `HUMI`), and compute robust, clustered standard errors. |
| **4. Supervised learning block** | Danish | Implement the non-shuffled chronological dataset split (training on 2010–2014, validation on 2015). Train and optimize a non-linear Random Forest Regressor to capture physical weather-smog interactions, and compute holdout performance statistics (RMSE, MAE, $R^2$). |
| **5. Unsupervised / generative block** | Oliver | Execute K-Means clustering optimization across a range of $K \in [2,8]$. Identify the geometric "elbow" point to establish 4 distinct micro-climate regimes, and map clusters back to localized historical PM2.5 distributions for domain validation. |
| **6. Synthesis & write-up** | Oliver | Execute Section 5 causal validation checks (Placebo Treatment and Random Common Cause refuters). Benchmark the machine learning model against OLS linear and Lag-1 baseline models. Cross-reference predictive vs. structural insights and finalize the proposal. |


---
## 7. Results *(complete for final submission)*


### 7a. Causal Inference

In [ ]:
# Causal inference analysis

### 7b. Supervised Learning

In [ ]:
# Supervised learning analysis

### 7c. Unsupervised / Generative

In [ ]:
# Unsupervised / generative analysis

## 8. Discussion & Conclusion *(complete for final submission)*

*Synthesise findings across all three method blocks. What does each lens reveal that the others miss? What are the limitations of your analysis?*
